#Made by Waranon Kaewket

#Install libraries & Import

In [ ]:
!pip install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu118 -q
!pip install ultralytics -q
!pip install insightface onnxruntime-gpu -q
!pip install opencv-python matplotlib scikit-learn -q
!pip install mediapipe -q
!pip install huggingface_hub -q
!wget -O pose_landmarker_full.task https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/1/pose_landmarker_full.task

--2026-03-07 18:37:51--  https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/1/pose_landmarker_full.task
Resolving storage.googleapis.com (storage.googleapis.com)... 173.194.203.207, 142.250.107.207, 173.194.202.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|173.194.203.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9398198 (9.0M) [application/octet-stream]
Saving to: ‘pose_landmarker_full.task’

pose_landmarker_ful 100%[===================>]   8.96M  55.3MB/s    in 0.2s    

2026-03-07 18:37:52 (55.3 MB/s) - ‘pose_landmarker_full.task’ saved [9398198/9398198]



In [ ]:
import cv2
import numpy as np
import time
import os
from collections import defaultdict
from insightface.app import FaceAnalysis
from ultralytics import YOLO
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

#Uploading Image & Videos

In [ ]:
# Cell 0: Upload files
from google.colab import files
import os

print("📁 กรุณาอัปโหลดไฟล์ต่อไปนี้:")
print("1. staff_face.jpg - ภาพใบหน้าพนักงาน")
print("2. Uniform.jpg - ภาพชุดฟอร์มพนักงานเต็มตัว")
print("3. employee_walk.mp4 - วิดีโอพนักงานเดิน")
print("4. intruder_walk.mp4 - วิดีโอผู้บุกรุก")

uploaded = files.upload()

# เช็คว่าอัปโหลดครบหรือยัง
required_files = ['staff_face.jpg', 'Uniform.jpg',
                  'Employee_Walk.mp4', 'Intruder_walk.mp4']
missing = [f for f in required_files if f not in uploaded.keys()]

if missing:
    print(f"⚠️ ยังขาดไฟล์: {missing}")
else:
    print("✅ อัปโหลดครบแล้ว!")

📁 กรุณาอัปโหลดไฟล์ต่อไปนี้:
1. staff_face.jpg - ภาพใบหน้าพนักงาน
2. Uniform.jpg - ภาพชุดฟอร์มพนักงานเต็มตัว
3. employee_walk.mp4 - วิดีโอพนักงานเดิน
4. intruder_walk.mp4 - วิดีโอผู้บุกรุก


Saving Employee_Walk.mp4 to Employee_Walk.mp4
Saving Intruder_walk.mp4 to Intruder_walk.mp4
Saving staff_face.jpg to staff_face.jpg
Saving Uniform.jpg to Uniform.jpg
✅ อัปโหลดครบแล้ว!


#Config of System

In [ ]:
class Config:
    # Face Recognition
    face_model = "buffalo_l"
    face_det_size = (640, 640)
    face_threshold = 0.3
    recognition_threshold = 0.1

    # Person Detection (YOLO)
    person_model_path = "yolov8n.pt"
    person_conf_threshold = 0.5

    # Clothing Detection (kesimeg YOLO)
    clothing_model_path = None  # Will be set after download
    clothing_conf_threshold = 0.5
    use_clothing_detection = True  # Enable/disable clothing detection

    # MediaPipe Pose
    mediapipe_model_path = "pose_landmarker_full.task"
    mediapipe_confidence = 0.5

    # Tracking
    tracking_iou_threshold = 0.3
    track_max_age = 30

    # Frame Processing
    frame_skip = 3  # Process every 3rd frame

    # Uniform Profile Matching
    uniform_hue_tolerance = 80      # ± degrees (รองรับแสงต่างกัน)
    uniform_sat_tolerance = 80      # ± value
    uniform_val_tolerance = 80      # ± value
    uniform_match_threshold = 0.6    # Overall match score threshold

    # Uniform matching weights
    weight_hue = 0.6  # Hue most important
    weight_sat = 0.2  # Saturation
    weight_val = 0.2  # Value

    # ROI Polygon (default)
    roi_polygon = np.array([
        [375, 480],
        [520, 255],
        [590, 260],
        [600, 480]
    ], dtype=np.int32)

    # Display
    display_fps = True
    display_names = True
    display_roi = True

    # MediaPipe Pose Landmarks Indices
    IDX_LEFT_SHOULDER = 11
    IDX_RIGHT_SHOULDER = 12
    IDX_LEFT_HIP = 23
    IDX_RIGHT_HIP = 24
    IDX_LEFT_KNEE = 25
    IDX_RIGHT_KNEE = 26

config = Config()

#System setup

In [ ]:
# ==========================================
# 1. MediaPipe Pose Setup
# ==========================================
def setup_mediapipe():
    """Setup MediaPipe Pose Landmarker"""
    if not os.path.exists(config.mediapipe_model_path):
        print("⬇️ Downloading MediaPipe Pose model...")
        os.system(f'wget -q -O {config.mediapipe_model_path} https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/1/pose_landmarker_full.task')

    BaseOptions = mp.tasks.BaseOptions
    PoseLandmarker = mp.tasks.vision.PoseLandmarker
    PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
    VisionRunningMode = mp.tasks.vision.RunningMode

    options = PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=config.mediapipe_model_path),
        running_mode=VisionRunningMode.IMAGE,
        min_pose_detection_confidence=config.mediapipe_confidence
    )

    return vision.PoseLandmarker.create_from_options(options)

In [ ]:
# ==========================================
# 2. Draw Pose Landmarks
# ==========================================

def draw_landmarks_manual(image, detection_result):
    """Draw pose landmarks on image"""
    if not detection_result.pose_landmarks:
        return image

    annotated = image.copy()
    h, w = image.shape[:2]

    # Skeleton connections
    connections = [
        (11, 12), (11, 23), (12, 24), (23, 24),  # Torso
        (11, 13), (13, 15), (12, 14), (14, 16),  # Arms
        (23, 25), (25, 27), (24, 26), (26, 28)   # Legs
    ]

    landmarks = detection_result.pose_landmarks[0]

    # Draw points
    for lm in landmarks:
        cx, cy = int(lm.x * w), int(lm.y * h)
        cv2.circle(annotated, (cx, cy), 4, (0, 0, 255), -1)

    # Draw lines
    for start_idx, end_idx in connections:
        start = landmarks[start_idx]
        end = landmarks[end_idx]
        p1 = (int(start.x * w), int(start.y * h))
        p2 = (int(end.x * w), int(end.y * h))
        cv2.line(annotated, p1, p2, (0, 255, 0), 2)

    return annotated

In [ ]:
# ==========================================
# 3. Frame Enhancement
# ==========================================

def enhance_frame_quality(frame):
    """Enhance CCTV frame quality"""
    lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l_enhanced = clahe.apply(l)
    enhanced = cv2.cvtColor(cv2.merge((l_enhanced, a, b)), cv2.COLOR_LAB2BGR)

    gaussian = cv2.GaussianBlur(enhanced, (0, 0), 1.5)
    sharpened = cv2.addWeighted(enhanced, 1.3, gaussian, -0.3, 0)

    return sharpened

def enhance_small_face(face_img):
    """Enhance small face images for better recognition"""
    h, w = face_img.shape[:2]
    if h < 30 or h > 80:
        return face_img

    scale = 112 / h
    upscaled = cv2.resize(face_img, None, fx=scale, fy=scale,
                         interpolation=cv2.INTER_LANCZOS4)
    gaussian = cv2.GaussianBlur(upscaled, (0, 0), 3.0)
    sharpened = cv2.addWeighted(upscaled, 1.5, gaussian, -0.5, 0)

    return sharpened

In [ ]:
# ==========================================
# 4. Face Recognition
# ==========================================

class FaceRecognizer:
    def __init__(self):
        print("📥 Loading buffalo_l face recognition model...")
        self.face_app = FaceAnalysis(
            name=config.face_model,
            providers=['CUDAExecutionProvider', 'CPUExecutionProvider']
        )
        self.face_app.prepare(ctx_id=0, det_size=config.face_det_size)

        self.known_faces = defaultdict(list)
        print("✅ buffalo_l loaded")

    def register_from_image(self, name, image_path):
        """Register face from image file"""
        img = cv2.imread(image_path)
        if img is None:
            return False

        faces = self.face_app.get(img)
        if not faces:
            return False

        face = max(faces, key=lambda x: (x.bbox[2]-x.bbox[0]) * (x.bbox[3]-x.bbox[1]))
        emb = face.embedding.reshape(1, -1)
        self.known_faces[name].append(emb)
        return True

    def recognize(self, person_crop):
        """Recognize face in person crop"""
        enhanced = enhance_small_face(person_crop)

        faces = self.face_app.get(enhanced)
        if not faces:
            return None, 0.0

        emb = faces[0].embedding.reshape(1, -1)
        best_score = 0.0
        best_name = None

        for name, embs in self.known_faces.items():
            for saved_emb in embs:
                score = np.dot(emb, saved_emb.T)[0][0] / (
                    np.linalg.norm(emb) * np.linalg.norm(saved_emb)
                )
                if score > best_score:
                    best_score = score
                    best_name = name

        return best_name, best_score


In [ ]:
# ==========================================
# 5. kesimeg Clothing Detector
# ==========================================

class ClothingDetector:
    """kesimeg YOLO clothing detector (4-class model)"""

    def __init__(self):
        if config.use_clothing_detection and config.clothing_model_path:
            print("📥 Loading kesimeg clothing detection model...")
            self.model = YOLO(config.clothing_model_path)
            print("✅ kesimeg clothing detector loaded")
            print(f"   Classes: {list(self.model.names.values())}")
        else:
            self.model = None
            print("⚠️  Clothing detection disabled")

    def detect(self, frame, person_bbox=None):
        """
        Detect clothing in frame or within person bbox
        Returns: {'clothing': {'bbox': [x1,y1,x2,y2], 'confidence': 0.9}}
        """
        if self.model is None:
            return {}

        # Crop to person region if provided
        if person_bbox:
            x1, y1, x2, y2 = person_bbox
            roi = frame[y1:y2, x1:x2]
            offset_x, offset_y = x1, y1
        else:
            roi = frame
            offset_x, offset_y = 0, 0

        # Run YOLO
        results = self.model(roi, conf=config.clothing_conf_threshold, verbose=False)

        clothing = {}
        for result in results:
            for box in result.boxes:
                cls = int(box.cls[0])
                conf = float(box.conf[0])
                bbox = box.xyxy[0].cpu().numpy().astype(int)

                class_name = result.names[cls]

                # We only care about 'clothing' class
                if class_name.lower() == 'clothing':
                    # Adjust bbox to frame coordinates
                    bbox[0] += offset_x
                    bbox[1] += offset_y
                    bbox[2] += offset_x
                    bbox[3] += offset_y

                    # Store with highest confidence
                    if 'clothing' not in clothing or clothing['clothing']['confidence'] < conf:
                        clothing['clothing'] = {
                            'bbox': bbox.tolist(),
                            'confidence': conf
                        }

        return clothing

In [ ]:
# ==========================================
# 6. Profile-based Uniform Checker
# ==========================================

class UniformChecker:
    """Profile-based uniform checking using average HSV color"""

    def __init__(self, clothing_detector):
        self.clothing_detector = clothing_detector
        self.profiles = {}  # {profile_name: {hue_mean, sat_mean, val_mean}}
        self.employee_profiles = {}  # {employee_name: profile_name}

    def create_profile_from_image(self, profile_name, image_path):
        """
        Create uniform profile from template image
        Returns: True if successful
        """
        img = cv2.imread(image_path)
        if img is None:
            print(f"   ❌ Cannot load image: {image_path}")
            return False

        print(f"   🔍 Creating profile '{profile_name}'...")

        # Detect clothing
        detections = self.clothing_detector.detect(img)

        if 'clothing' not in detections:
            print(f"   ❌ No clothing detected in template")
            return False

        bbox = detections['clothing']['bbox']
        x1, y1, x2, y2 = bbox

        # Extract clothing region
        clothing_roi = img[y1:y2, x1:x2]

        if clothing_roi.size == 0:
            print(f"   ❌ Empty clothing ROI")
            return False

        # Convert to HSV and calculate mean
        hsv = cv2.cvtColor(clothing_roi, cv2.COLOR_BGR2HSV)
        mean_hsv = cv2.mean(hsv)[:3]  # Average all pixels

        # Store profile
        self.profiles[profile_name] = {
            'hue_mean': mean_hsv[0],
            'sat_mean': mean_hsv[1],
            'val_mean': mean_hsv[2],
            'hue_tolerance': config.uniform_hue_tolerance,
            'sat_tolerance': config.uniform_sat_tolerance,
            'val_tolerance': config.uniform_val_tolerance,
            'match_threshold': config.uniform_match_threshold
        }

        print(f"   ✅ Profile created: {profile_name}")
        print(f"      Average HSV: H={mean_hsv[0]:.1f}, S={mean_hsv[1]:.1f}, V={mean_hsv[2]:.1f}")
        print(f"      Tolerance: ±{config.uniform_hue_tolerance}° hue, ±{config.uniform_sat_tolerance} sat/val")

        return True

    def assign_profile(self, employee_name, profile_name):
        """Assign uniform profile to employee"""
        self.employee_profiles[employee_name] = profile_name

    def check_uniform(self, person_crop, employee_name):
        """
        Check if person is wearing correct uniform
        Returns: (status, color, details)
        """
        # Get employee's profile
        profile_name = self.employee_profiles.get(employee_name)

        if not profile_name:
            return "No Profile", (128, 128, 128), {}

        if profile_name not in self.profiles:
            return "Profile Not Found", (128, 128, 128), {}

        profile = self.profiles[profile_name]

        # Detect clothing
        detections = self.clothing_detector.detect(person_crop)

        if 'clothing' not in detections:
            return "No Clothing Detected", (128, 128, 128), {}

        bbox = detections['clothing']['bbox']

        # Handle bbox coordinates
        h, w = person_crop.shape[:2]
        x1, y1, x2, y2 = bbox

        # Clip to bounds
        x1 = max(0, min(x1, w))
        y1 = max(0, min(y1, h))
        x2 = max(0, min(x2, w))
        y2 = max(0, min(y2, h))

        if x2 <= x1 or y2 <= y1:
            return "Invalid BBox", (128, 128, 128), {}

        # Extract clothing region
        clothing_roi = person_crop[y1:y2, x1:x2]

        if clothing_roi.size == 0:
            return "Empty ROI", (128, 128, 128), {}

        # Calculate average HSV
        hsv = cv2.cvtColor(clothing_roi, cv2.COLOR_BGR2HSV)
        mean_hsv = cv2.mean(hsv)[:3]

        # Compare with profile
        hue_diff = abs(mean_hsv[0] - profile['hue_mean'])
        if hue_diff > 90:
            hue_diff = 180 - hue_diff  # Wrap around

        sat_diff = abs(mean_hsv[1] - profile['sat_mean'])
        val_diff = abs(mean_hsv[2] - profile['val_mean'])

        # Calculate match scores (0-1)
        hue_match = max(0, 1.0 - (hue_diff / profile['hue_tolerance']))
        sat_match = max(0, 1.0 - (sat_diff / profile['sat_tolerance']))
        val_match = max(0, 1.0 - (val_diff / profile['val_tolerance']))

        # Weighted average
        overall_match = (
            hue_match * config.weight_hue +
            sat_match * config.weight_sat +
            val_match * config.weight_val
        )

        details = {
            'hue_diff': hue_diff,
            'sat_diff': sat_diff,
            'val_diff': val_diff,
            'match_score': overall_match,
            'actual_hsv': mean_hsv,
            'expected_hsv': (profile['hue_mean'], profile['sat_mean'], profile['val_mean'])
        }

        # Determine status
        if overall_match >= profile['match_threshold']:
            return "Correct Uniform", (0, 255, 0), details
        else:
            return "Wrong Uniform", (0, 0, 255), details


In [ ]:
# ==========================================
# 7. Suspicious Behavior Detection
# ==========================================

class BehaviorDetector:
    def __init__(self, pose_landmarker):
        self.landmarker = pose_landmarker

    def detect_suspicious_pose(self, person_crop):
        """
        Detect suspicious behaviors
        Returns: (label, color, annotated_image, behaviors)
        """
        img_rgb = cv2.cvtColor(person_crop, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
        results = self.landmarker.detect(mp_image)

        annotated = person_crop.copy()
        behaviors = []

        if not results.pose_landmarks:
            return "Intruder (No Pose)", (0, 0, 255), annotated, behaviors

        # Draw landmarks
        annotated = draw_landmarks_manual(annotated, results)

        lm = results.pose_landmarks[0]

        # Get key points
        left_shoulder_y = lm[config.IDX_LEFT_SHOULDER].y
        right_shoulder_y = lm[config.IDX_RIGHT_SHOULDER].y
        left_hip_y = lm[config.IDX_LEFT_HIP].y
        right_hip_y = lm[config.IDX_RIGHT_HIP].y
        left_knee_y = lm[config.IDX_LEFT_KNEE].y
        right_knee_y = lm[config.IDX_RIGHT_KNEE].y

        avg_shoulder_y = (left_shoulder_y + right_shoulder_y) / 2
        avg_hip_y = (left_hip_y + right_hip_y) / 2
        avg_knee_y = (left_knee_y + right_knee_y) / 2

        # Detect behaviors
        if avg_shoulder_y > 0.7:
            behaviors.append("Crawling")
        elif avg_knee_y < avg_hip_y:
            behaviors.append("Crouching")

        knee_hip_dist = abs(avg_knee_y - avg_hip_y)
        if knee_hip_dist < 0.15:
            behaviors.append("Running")

        if behaviors:
            label = f"Intruder ({', '.join(behaviors)})"
        else:
            label = "Intruder (Standing)"

        return label, (0, 0, 255), annotated, behaviors

In [ ]:
# ==========================================
# 8. Simple IoU Tracker
# ==========================================

class SimpleTracker:
    def __init__(self):
        self.tracks = {}
        self.next_id = 0

    @staticmethod
    def compute_iou(box1, box2):
        x1_1, y1_1, x2_1, y2_1 = box1
        x1_2, y1_2, x2_2, y2_2 = box2

        x1_i = max(x1_1, x1_2)
        y1_i = max(y1_1, y1_2)
        x2_i = min(x2_1, x2_2)
        y2_i = min(y2_1, y2_2)

        if x2_i < x1_i or y2_i < y1_i:
            return 0.0

        inter_area = (x2_i - x1_i) * (y2_i - y1_i)
        box1_area = (x2_1 - x1_1) * (y2_1 - y1_1)
        box2_area = (x2_2 - x1_2) * (y2_2 - y1_2)
        union_area = box1_area + box2_area - inter_area

        return inter_area / union_area if union_area > 0 else 0.0

    def update(self, detections, frame_count):
        # Remove old tracks
        to_remove = []
        for track_id, track in self.tracks.items():
            if frame_count - track['last_seen'] > config.track_max_age:
                to_remove.append(track_id)
        for track_id in to_remove:
            del self.tracks[track_id]

        # Match detections
        matched_detections = []

        for det in detections:
            det_bbox = det['bbox']
            best_iou = 0.0
            best_track_id = None

            for track_id, track in self.tracks.items():
                iou = self.compute_iou(det_bbox, track['bbox'])
                if iou > best_iou:
                    best_iou = iou
                    best_track_id = track_id

            if best_iou > config.tracking_iou_threshold:
                self.tracks[best_track_id]['bbox'] = det_bbox
                self.tracks[best_track_id]['last_seen'] = frame_count

                if self.tracks[best_track_id].get('name'):
                    det['name'] = self.tracks[best_track_id]['name']
                else:
                    self.tracks[best_track_id]['name'] = det.get('name')

                det['track_id'] = best_track_id
            else:
                det['track_id'] = self.next_id
                self.tracks[self.next_id] = {
                    'bbox': det_bbox,
                    'name': det.get('name'),
                    'last_seen': frame_count
                }
                self.next_id += 1

            matched_detections.append(det)

        return matched_detections

In [ ]:
# ==========================================
# 9. Main Surveillance System
# ==========================================

class SurveillanceSystem:
    def __init__(self):
        print("🚀 Initializing Surveillance System...")

        # Setup MediaPipe
        self.pose_landmarker = setup_mediapipe()

        # Initialize components
        self.face_recognizer = FaceRecognizer()
        self.clothing_detector = ClothingDetector()
        self.uniform_checker = UniformChecker(self.clothing_detector)
        self.behavior_detector = BehaviorDetector(self.pose_landmarker)
        self.tracker = SimpleTracker()

        # Load person detector
        print("📥 Loading YOLO person detector...")
        self.person_detector = YOLO(config.person_model_path)
        print("✅ YOLO loaded")

        self.frame_count = 0
        print("✅ System ready\n")

    def create_uniform_profile(self, profile_name, template_image_path):
        """
        Create a reusable uniform profile
        Can be assigned to multiple employees

        Args:
            profile_name: Unique name for this uniform (e.g., 'engineer', 'security')
            template_image_path: Path to template image

        Returns:
            bool: True if successful
        """
        if not config.use_clothing_detection:
            print(f"⚠️  Clothing detection is disabled")
            return False

        success = self.uniform_checker.create_profile_from_image(
            profile_name, template_image_path
        )

        if success:
            print(f"✅ Created reusable uniform profile: {profile_name}")
        else:
            print(f"❌ Failed to create uniform profile: {profile_name}")

        return success

    def register_employee(self, name, face_image_path, uniform_profile=None):
        """
        Register employee with face (and optionally assign uniform profile)

        Args:
            name: Employee name/ID
            face_image_path: Path to face image
            uniform_profile: (Optional) Name of uniform profile to assign

        Returns:
            bool: True if successful
        """
        # Register face
        success = self.face_recognizer.register_from_image(name, face_image_path)
        if not success:
            print(f"❌ Failed to register face: {name}")
            return False

        print(f"✅ Registered face: {name}")

        # Assign uniform profile if provided
        if uniform_profile:
            self.assign_uniform(name, uniform_profile)

        return True

    def assign_uniform(self, employee_name, profile_name):
        """
        Assign a uniform profile to an employee

        Args:
            employee_name: Employee name/ID
            profile_name: Uniform profile name

        Returns:
            bool: True if successful
        """
        if profile_name not in self.uniform_checker.profiles:
            print(f"❌ Uniform profile '{profile_name}' does not exist")
            print(f"   Available profiles: {list(self.uniform_checker.profiles.keys())}")
            return False

        self.uniform_checker.assign_profile(employee_name, profile_name)
        print(f"✅ Assigned uniform: {employee_name} → {profile_name}")
        return True

    def set_roi(self, polygon):
        """Set ROI polygon"""
        config.roi_polygon = np.array(polygon, dtype=np.int32)

    def process_frame(self, frame):
        """Process a single frame"""
        self.frame_count += 1

        # Skip frames
        if self.frame_count % config.frame_skip != 0:
            return frame, None

        start_time = time.time()

        # Enhance frame
        enhanced = enhance_frame_quality(frame)

        # Detect persons
        results = self.person_detector(enhanced, classes=[0], verbose=False)

        detections = []

        for box in results[0].boxes.xyxy.cpu().numpy():
            x1, y1, x2, y2 = map(int, box)

            # Check if in ROI
            cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
            if cv2.pointPolygonTest(config.roi_polygon, (cx, cy), False) < 0:
                continue

            # Crop person
            person_crop = enhanced[y1:y2, x1:x2]
            if person_crop.size == 0:
                continue

            # Face recognition
            name, score = self.face_recognizer.recognize(person_crop)

            detection = {
                'bbox': [x1, y1, x2, y2],
                'name': name if score > config.recognition_threshold else 'Unknown',
                'face_score': score,
                'pose_crop': None
            }

            if score > config.recognition_threshold:
                # Known employee → check uniform
                if config.use_clothing_detection:
                    status, color, details = self.uniform_checker.check_uniform(
                        person_crop, name
                    )
                    detection['uniform_status'] = status
                    detection['color'] = color
                    detection['match_score'] = details.get('match_score', 0.0)
                else:
                    detection['uniform_status'] = "Check Disabled"
                    detection['color'] = (128, 128, 128)
            else:
                # Unknown person → detect suspicious behavior
                label, color, pose_crop, behaviors = self.behavior_detector.detect_suspicious_pose(
                    person_crop
                )
                detection['uniform_status'] = label
                detection['color'] = color
                detection['pose_crop'] = pose_crop
                detection['behaviors'] = behaviors

            detections.append(detection)

        # Track detections
        tracked = self.tracker.update(detections, self.frame_count)

        # Annotate frame
        annotated = self._annotate_frame(frame.copy(), tracked)

        fps = 1.0 / (time.time() - start_time)

        if config.display_fps:
            cv2.putText(annotated, f"FPS: {fps:.1f}", (10, 30),
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        return annotated, {
            'detections': tracked,
            'fps': fps,
            'frame_count': self.frame_count
        }

    def _annotate_frame(self, frame, detections):
        """Draw annotations on frame"""
        # Draw ROI
        if config.display_roi:
            cv2.polylines(frame, [config.roi_polygon], True, (255, 0, 0), 2)

        for det in detections:
            x1, y1, x2, y2 = det['bbox']
            color = det.get('color', (255, 255, 0))

            # Draw bbox
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

            # Draw label
            if config.display_names:
                label = det.get('name', 'Unknown')
                if label != 'Unknown':
                    score = det.get('face_score', 0.0)
                    match = det.get('match_score', 0.0)
                    label += f" ({score:.2f}|{match:.2f})"

                cv2.putText(frame, label, (x1, y1-10),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

            # Draw uniform status
            status = det.get('uniform_status', '')
            cv2.putText(frame, status, (x1, y2+20),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

            # Overlay pose crop if available
            if det.get('pose_crop') is not None:
                pose_crop = det['pose_crop']
                ph, pw = pose_crop.shape[:2]
                if y1+ph < frame.shape[0] and x1+pw < frame.shape[1]:
                    frame[y1:y1+ph, x1:x1+pw] = pose_crop

        return frame

    def process_video(self, video_path, output_path=None, display=True):
        """Process video file"""
        cap = cv2.VideoCapture(video_path)

        if not cap.isOpened():
            print(f"❌ Cannot open video: {video_path}")
            return

        fps = int(cap.get(cv2.CAP_PROP_FPS)) or 25
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

        print(f"📹 Video: {width}x{height} @ {fps} FPS")

        writer = None
        if output_path:
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

        try:
            while True:
                ret, frame = cap.read()
                if not ret:
                    break

                annotated, results = self.process_frame(frame)

                if writer and results:
                    writer.write(annotated)

                if display:
                    cv2.imshow('Surveillance System', annotated)
                    if cv2.waitKey(1) & 0xFF == ord('q'):
                        break

        finally:
            cap.release()
            if writer:
                writer.release()

            # Only destroy windows if display was enabled
            if display:
                try:
                    cv2.destroyAllWindows()
                except:
                    pass  # Ignore errors in headless environments

            print(f"\n✅ Processed {self.frame_count} frames")

In [ ]:
# ==========================================
# 10. Colab Video Display Helpers
# ==========================================

class ColabVideoHelpers:
    """
    Utilities for displaying and converting videos in Google Colab
    - HTML5 video player with base64 encoding
    - FFmpeg H.264/H.265 conversion for better compression
    """

    @staticmethod
    def display_video(video_path, width=800):
        """
        Display video in Colab using HTML5 video player

        Args:
            video_path: Path to video file
            width: Video width in pixels (default 800)

        Returns:
            HTML object for display
        """
        try:
            from IPython.display import HTML
            from base64 import b64encode

            # Read video file
            with open(video_path, "rb") as f:
                video_file = f.read()

            # Encode to base64
            video_url = f"data:video/mp4;base64,{b64encode(video_file).decode()}"

            file_size_mb = len(video_file) / 1024 / 1024
            print(f"📹 Displaying video: {video_path}")
            print(f"   File size: {file_size_mb:.1f} MB")

            # Create HTML video player
            html = f"""
            <video width="{width}" controls>
                <source src="{video_url}" type="video/mp4">
                Your browser does not support the video tag.
            </video>
            """

            return HTML(html)

        except ImportError:
            print("❌ IPython not available (not in Colab?)")
            return None
        except FileNotFoundError:
            print(f"❌ Video not found: {video_path}")
            return None
        except Exception as e:
            print(f"❌ Error displaying video: {e}")
            return None

    @staticmethod
    def convert_to_h264(input_path, output_path=None, crf=23, preset='medium'):
        """
        Convert video to H.264 (MP4) for web playback and smaller file size

        Args:
            input_path: Input video path
            output_path: Output path (default: input_h264.mp4)
            crf: Quality (0-51, lower=better, default 23)
            preset: Encoding speed (ultrafast, fast, medium, slow, veryslow)

        Returns:
            str: Output path if successful, None otherwise
        """
        import subprocess

        if output_path is None:
            base, ext = os.path.splitext(input_path)
            output_path = f"{base}_h264.mp4"

        print(f"🔄 Converting to H.264...")
        print(f"   Input: {input_path}")
        print(f"   Output: {output_path}")
        print(f"   Quality: CRF {crf}, Preset: {preset}")

        try:
            # Get input file size
            input_size = os.path.getsize(input_path) / 1024 / 1024
            print(f"   Input size: {input_size:.1f} MB")

            # FFmpeg command
            command = [
                'ffmpeg', '-y',
                '-loglevel', 'error',
                '-i', input_path,
                '-c:v', 'libx264',      # H.264 codec
                '-crf', str(crf),        # Quality
                '-preset', preset,       # Speed/compression trade-off
                '-c:a', 'aac',           # Audio codec
                '-b:a', '128k',          # Audio bitrate
                '-movflags', '+faststart',  # Enable streaming
                output_path
            ]

            subprocess.run(command, check=True, capture_output=True)

            # Get output file size
            output_size = os.path.getsize(output_path) / 1024 / 1024
            compression = (1 - output_size / input_size) * 100

            print(f"✅ Conversion successful!")
            print(f"   Output size: {output_size:.1f} MB")
            print(f"   Compression: {compression:.1f}%")

            return output_path

        except subprocess.CalledProcessError as e:
            print(f"❌ FFmpeg conversion failed")
            if e.stderr:
                print(f"   Error: {e.stderr.decode()}")
            return None
        except Exception as e:
            print(f"❌ Error: {e}")
            return None

    @staticmethod
    def convert_to_h265(input_path, output_path=None, crf=28, preset='medium'):
        """
        Convert video to H.265 (HEVC) for maximum compression
        Note: Better compression than H.264 but slower encoding

        Args:
            input_path: Input video path
            output_path: Output path (default: input_h265.mp4)
            crf: Quality (0-51, lower=better, default 28 for H.265)
            preset: Encoding speed

        Returns:
            str: Output path if successful, None otherwise
        """
        import subprocess

        if output_path is None:
            base, ext = os.path.splitext(input_path)
            output_path = f"{base}_h265.mp4"

        print(f"🔄 Converting to H.265 (HEVC)...")
        print(f"   Input: {input_path}")
        print(f"   Output: {output_path}")
        print(f"   Quality: CRF {crf}, Preset: {preset}")
        print(f"   ⚠️  H.265 encoding is slower but gives better compression")

        try:
            input_size = os.path.getsize(input_path) / 1024 / 1024
            print(f"   Input size: {input_size:.1f} MB")

            command = [
                'ffmpeg', '-y',
                '-loglevel', 'error',
                '-i', input_path,
                '-c:v', 'libx265',      # H.265 codec
                '-crf', str(crf),
                '-preset', preset,
                '-c:a', 'aac',
                '-b:a', '128k',
                '-tag:v', 'hvc1',       # For better compatibility
                '-movflags', '+faststart',
                output_path
            ]

            subprocess.run(command, check=True, capture_output=True)

            output_size = os.path.getsize(output_path) / 1024 / 1024
            compression = (1 - output_size / input_size) * 100

            print(f"✅ Conversion successful!")
            print(f"   Output size: {output_size:.1f} MB")
            print(f"   Compression: {compression:.1f}%")

            return output_path

        except subprocess.CalledProcessError as e:
            print(f"❌ FFmpeg conversion failed")
            if e.stderr:
                print(f"   Error: {e.stderr.decode()}")
            return None
        except Exception as e:
            print(f"❌ Error: {e}")
            return None

    @staticmethod
    def display_and_download(video_path, convert=True, codec='h264', crf=23, width=800):
        """
        All-in-one: Convert (optional), display, and download video

        Args:
            video_path: Path to video
            convert: Whether to convert to H.264/H.265 (default True)
            codec: 'h264' or 'h265' (default 'h264')
            crf: Quality setting (default 23)
            width: Display width (default 800)

        Returns:
            HTML object for display
        """
        display_path = video_path

        # Convert if requested
        if convert:
            if codec == 'h265':
                converted = ColabVideoHelpers.convert_to_h265(video_path, crf=crf)
            else:
                converted = ColabVideoHelpers.convert_to_h264(video_path, crf=crf)

            if converted:
                display_path = converted
                print(f"\n📹 Using converted video for display")

        # Display
        print(f"\n📺 Displaying video...")
        html = ColabVideoHelpers.display_video(display_path, width=width)

        # Offer download
        print(f"\n💾 To download, run:")
        print(f"   from google.colab import files")
        print(f"   files.download('{display_path}')")

        return html

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import matplotlib.patches as mpatches

class UniformColorDebugger:
    """Debug tool for uniform color detection"""

    def __init__(self, system):
        """
        Args:
            system: SurveillanceSystem instance
        """
        self.system = system
        self.clothing_detector = system.clothing_detector
        self.uniform_checker = system.uniform_checker

    def debug_single_image(self, image_path, show_plot=True):
        """
        Debug uniform detection on a single image

        Args:
            image_path: Path to image
            show_plot: Whether to display plot

        Returns:
            dict with debug info
        """
        # Load image
        img = cv2.imread(image_path)
        if img is None:
            print(f"❌ Cannot load image: {image_path}")
            return None

        print(f"🔍 Debugging image: {image_path}")
        print(f"   Size: {img.shape[1]}x{img.shape[0]}")

        # Detect clothing
        detections = self.clothing_detector.detect(img)

        if 'clothing' not in detections:
            print(f"❌ No clothing detected")
            return None

        bbox = detections['clothing']['bbox']
        conf = detections['clothing']['confidence']

        print(f"\n📦 Clothing Detection:")
        print(f"   BBox: {bbox}")
        print(f"   Confidence: {conf:.2f}")

        # Extract ROI
        x1, y1, x2, y2 = bbox
        clothing_roi = img[y1:y2, x1:x2]

        if clothing_roi.size == 0:
            print(f"❌ Empty clothing ROI")
            return None

        # Convert to HSV and calculate mean
        hsv = cv2.cvtColor(clothing_roi, cv2.COLOR_BGR2HSV)
        mean_hsv = cv2.mean(hsv)[:3]

        # Get color histogram
        hist_h = cv2.calcHist([hsv], [0], None, [180], [0, 180])
        hist_s = cv2.calcHist([hsv], [1], None, [256], [0, 256])
        hist_v = cv2.calcHist([hsv], [2], None, [256], [0, 256])

        print(f"\n🎨 Detected HSV:")
        print(f"   Hue: {mean_hsv[0]:.1f}°")
        print(f"   Saturation: {mean_hsv[1]:.1f}")
        print(f"   Value: {mean_hsv[2]:.1f}")

        # Convert HSV to RGB for display
        mean_bgr = cv2.cvtColor(
            np.uint8([[mean_hsv]]),
            cv2.COLOR_HSV2BGR
        )[0][0]
        mean_rgb = cv2.cvtColor(
            np.uint8([[mean_bgr]]),
            cv2.COLOR_BGR2RGB
        )[0][0]

        print(f"\n🖍️  Representative Color:")
        print(f"   RGB: ({mean_rgb[0]}, {mean_rgb[1]}, {mean_rgb[2]})")

        # Compare with all profiles
        print(f"\n📊 Profile Comparisons:")

        profile_matches = {}
        for profile_name, profile in self.uniform_checker.profiles.items():
            hue_diff = abs(mean_hsv[0] - profile['hue_mean'])
            if hue_diff > 90:
                hue_diff = 180 - hue_diff

            sat_diff = abs(mean_hsv[1] - profile['sat_mean'])
            val_diff = abs(mean_hsv[2] - profile['val_mean'])

            hue_match = max(0, 1.0 - (hue_diff / profile['hue_tolerance']))
            sat_match = max(0, 1.0 - (sat_diff / profile['sat_tolerance']))
            val_match = max(0, 1.0 - (val_diff / profile['val_tolerance']))

            # Weighted average
            from integrated_surveillance_system_v2_kesimeg import config
            overall_match = (
                hue_match * config.weight_hue +
                sat_match * config.weight_sat +
                val_match * config.weight_val
            )

            profile_matches[profile_name] = {
                'hue_diff': hue_diff,
                'sat_diff': sat_diff,
                'val_diff': val_diff,
                'overall_match': overall_match,
                'profile_hsv': (profile['hue_mean'], profile['sat_mean'], profile['val_mean'])
            }

            status = "✅ MATCH" if overall_match >= profile['match_threshold'] else "❌ NO MATCH"

            print(f"\n   Profile: {profile_name} {status}")
            print(f"      Expected HSV: H={profile['hue_mean']:.1f}, S={profile['sat_mean']:.1f}, V={profile['val_mean']:.1f}")
            print(f"      Differences: ΔH={hue_diff:.1f}°, ΔS={sat_diff:.1f}, ΔV={val_diff:.1f}")
            print(f"      Match Score: {overall_match:.2f} (threshold: {profile['match_threshold']:.2f})")

        # Visualize
        if show_plot:
            self._plot_debug_visualization(
                img, bbox, clothing_roi, mean_hsv, mean_rgb,
                hist_h, hist_s, hist_v, profile_matches
            )

        return {
            'bbox': bbox,
            'mean_hsv': mean_hsv,
            'mean_rgb': mean_rgb,
            'profile_matches': profile_matches
        }

    def debug_video_frame(self, video_path, frame_number, show_plot=True):
        """
        Debug specific frame from video

        Args:
            video_path: Path to video
            frame_number: Frame to extract
            show_plot: Whether to display plot
        """
        cap = cv2.VideoCapture(video_path)

        # Set frame position
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
        ret, frame = cap.read()
        cap.release()

        if not ret:
            print(f"❌ Cannot read frame {frame_number}")
            return None

        # Save temporary image
        temp_path = f"/tmp/debug_frame_{frame_number}.jpg"
        cv2.imwrite(temp_path, frame)

        print(f"📹 Video: {video_path}")
        print(f"🎬 Frame: {frame_number}")

        # Debug the frame
        return self.debug_single_image(temp_path, show_plot=show_plot)

    def _plot_debug_visualization(self, img, bbox, clothing_roi, mean_hsv, mean_rgb,
                                   hist_h, hist_s, hist_v, profile_matches):
        """Plot comprehensive debug visualization"""

        # Create figure
        fig = plt.figure(figsize=(16, 10))

        # 1. Original image with bbox
        ax1 = plt.subplot(3, 4, 1)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax1.imshow(img_rgb)
        x1, y1, x2, y2 = bbox
        rect = Rectangle((x1, y1), x2-x1, y2-y1,
                         linewidth=2, edgecolor='red', facecolor='none')
        ax1.add_patch(rect)
        ax1.set_title('Original Image\n(Red = Detected Clothing)')
        ax1.axis('off')

        # 2. Clothing ROI
        ax2 = plt.subplot(3, 4, 2)
        clothing_rgb = cv2.cvtColor(clothing_roi, cv2.COLOR_BGR2RGB)
        ax2.imshow(clothing_rgb)
        ax2.set_title('Clothing Region (Extracted)')
        ax2.axis('off')

        # 3. Average color swatch
        ax3 = plt.subplot(3, 4, 3)
        color_swatch = np.ones((100, 100, 3), dtype=np.uint8)
        color_swatch[:, :] = mean_rgb
        ax3.imshow(color_swatch)
        ax3.set_title(f'Average Color\nRGB: {mean_rgb}\nHSV: ({mean_hsv[0]:.0f}, {mean_hsv[1]:.0f}, {mean_hsv[2]:.0f})')
        ax3.axis('off')

        # 4. HSV visualization
        ax4 = plt.subplot(3, 4, 4)
        hsv_img = cv2.cvtColor(clothing_roi, cv2.COLOR_BGR2HSV)
        ax4.imshow(hsv_img)
        ax4.set_title('HSV Representation')
        ax4.axis('off')

        # 5. Hue histogram
        ax5 = plt.subplot(3, 4, 5)
        ax5.plot(hist_h, color='red')
        ax5.axvline(mean_hsv[0], color='blue', linestyle='--', label=f'Mean: {mean_hsv[0]:.1f}°')
        ax5.set_title('Hue Distribution')
        ax5.set_xlabel('Hue (0-179)')
        ax5.set_ylabel('Frequency')
        ax5.legend()
        ax5.grid(True, alpha=0.3)

        # 6. Saturation histogram
        ax6 = plt.subplot(3, 4, 6)
        ax6.plot(hist_s, color='green')
        ax6.axvline(mean_hsv[1], color='blue', linestyle='--', label=f'Mean: {mean_hsv[1]:.1f}')
        ax6.set_title('Saturation Distribution')
        ax6.set_xlabel('Saturation (0-255)')
        ax6.set_ylabel('Frequency')
        ax6.legend()
        ax6.grid(True, alpha=0.3)

        # 7. Value histogram
        ax7 = plt.subplot(3, 4, 7)
        ax7.plot(hist_v, color='blue')
        ax7.axvline(mean_hsv[2], color='red', linestyle='--', label=f'Mean: {mean_hsv[2]:.1f}')
        ax7.set_title('Value (Brightness) Distribution')
        ax7.set_xlabel('Value (0-255)')
        ax7.set_ylabel('Frequency')
        ax7.legend()
        ax7.grid(True, alpha=0.3)

        # 8-11. Profile comparisons
        for idx, (profile_name, match_data) in enumerate(profile_matches.items()):
            ax = plt.subplot(3, 4, 8 + idx)

            # Create comparison bars
            categories = ['Hue', 'Sat', 'Val']
            detected = list(mean_hsv)
            profile = list(match_data['profile_hsv'])

            x = np.arange(len(categories))
            width = 0.35

            bars1 = ax.bar(x - width/2, detected, width, label='Detected', alpha=0.8)
            bars2 = ax.bar(x + width/2, profile, width, label='Profile', alpha=0.8)

            ax.set_title(f'{profile_name}\nMatch: {match_data["overall_match"]:.2%}')
            ax.set_ylabel('Value')
            ax.set_xticks(x)
            ax.set_xticklabels(categories)
            ax.legend()
            ax.grid(True, alpha=0.3, axis='y')

            # Color code based on match
            if match_data['overall_match'] >= 0.45:
                ax.patch.set_facecolor('#90EE90')  # Light green
            else:
                ax.patch.set_facecolor('#FFB6C6')  # Light red

        plt.tight_layout()
        plt.savefig('/tmp/uniform_debug.png', dpi=150, bbox_inches='tight')
        print(f"\n💾 Debug visualization saved: /tmp/uniform_debug.png")

        try:
            plt.show()
        except:
            print("   (Display not available, file saved instead)")

    def compare_template_vs_video(self, template_path, video_path, frame_number):
        """
        Compare template image vs video frame side-by-side

        Args:
            template_path: Path to template image
            video_path: Path to video
            frame_number: Frame to compare
        """
        print("=" * 60)
        print("TEMPLATE vs VIDEO COMPARISON")
        print("=" * 60)

        print("\n📋 TEMPLATE IMAGE:")
        template_result = self.debug_single_image(template_path, show_plot=False)

        print("\n" + "=" * 60)
        print("\n📹 VIDEO FRAME:")
        video_result = self.debug_video_frame(video_path, frame_number, show_plot=False)

        if template_result and video_result:
            print("\n" + "=" * 60)
            print("📊 COMPARISON SUMMARY:")
            print("=" * 60)

            t_hsv = template_result['mean_hsv']
            v_hsv = video_result['mean_hsv']

            hue_diff = abs(t_hsv[0] - v_hsv[0])
            if hue_diff > 90:
                hue_diff = 180 - hue_diff

            sat_diff = abs(t_hsv[1] - v_hsv[1])
            val_diff = abs(t_hsv[2] - v_hsv[2])

            print(f"\nTemplate HSV: H={t_hsv[0]:.1f}, S={t_hsv[1]:.1f}, V={t_hsv[2]:.1f}")
            print(f"Video HSV:    H={v_hsv[0]:.1f}, S={v_hsv[1]:.1f}, V={v_hsv[2]:.1f}")
            print(f"\nDifferences:")
            print(f"   ΔHue:        {hue_diff:.1f}°")
            print(f"   ΔSaturation: {sat_diff:.1f}")
            print(f"   ΔValue:      {val_diff:.1f}")

            # Side-by-side color swatches
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

            # Template color
            template_swatch = np.ones((200, 200, 3), dtype=np.uint8)
            template_swatch[:, :] = template_result['mean_rgb']
            ax1.imshow(template_swatch)
            ax1.set_title(f'Template Color\nHSV: ({t_hsv[0]:.0f}, {t_hsv[1]:.0f}, {t_hsv[2]:.0f})')
            ax1.axis('off')

            # Video color
            video_swatch = np.ones((200, 200, 3), dtype=np.uint8)
            video_swatch[:, :] = video_result['mean_rgb']
            ax2.imshow(video_swatch)
            ax2.set_title(f'Video Color\nHSV: ({v_hsv[0]:.0f}, {v_hsv[1]:.0f}, {v_hsv[2]:.0f})')
            ax2.axis('off')

            plt.tight_layout()
            plt.savefig('/tmp/template_vs_video_colors.png', dpi=150)
            print(f"\n💾 Color comparison saved: /tmp/template_vs_video_colors.png")

            try:
                plt.show()
            except:
                pass

#Main Method

In [ ]:
if __name__ == "__main__":
    """
    Example workflow with decoupled uniform profiles
    """
    # Note: Set config.clothing_model_path before initializing
    # Example:
    from huggingface_hub import hf_hub_download
    config.clothing_model_path = hf_hub_download(
    repo_id="kesimeg/yolov8n-clothing-detection",
    filename="best.pt"
    )

    system = SurveillanceSystem()

    # ==========================================
    # Method 1: Create reusable uniform profiles
    # ==========================================

    # Step 1: Create uniform profiles (once)
    system.create_uniform_profile("staff_uniform", "Uniform.jpg")

    # Step 2: Register employees
    system.register_employee("Froy", "staff_face.jpg")
    # Step 3: Assign uniforms
    system.assign_uniform("Froy", "staff_uniform")
    # ==========================================
    # Method 2: Quick registration (backward compatible)
    # ==========================================

    # If each person has unique uniform, you can still do:
    #system.create_uniform_profile("staff_uniform", "staff_template.jpg")
    #system.register_employee("Staff", "staff_face.jpg", uniform_profile="staff_uniform")

    # ==========================================
    # Process video
    # ==========================================

    system.process_video("Employee_Walk.mp4", "output.mp4", display=False)

    # ==========================================
    # Display video in Colab (optional)
    # ==========================================

    # Option 1: Display directly
    # html = ColabVideoHelpers.display_video("output.mp4", width=800)
    # from IPython.display import display
    # display(html)

    # Option 2: Convert to H.264 then display (recommended)
    compressed = ColabVideoHelpers.convert_to_h264("output.mp4", crf=23)
    if compressed:
      html = ColabVideoHelpers.display_video(compressed)
      display(html)

    # Option 3: All-in-one (easiest)
    # html = ColabVideoHelpers.display_and_download("output.mp4", convert=True, codec='h264')
    # display(html)
    system.process_video("Intruder_walk.mp4", "output2.mp4", display=False)
    compressed = ColabVideoHelpers.convert_to_h264("output2.mp4", crf=23)
    if compressed:
      html = ColabVideoHelpers.display_video(compressed)
      display(html)
    print("\n✅ Processing complete!")
    print("\nTo display in Colab:")
    print("  from IPython.display import display")
    print("  html = ColabVideoHelpers.display_and_download('output.mp4')")
    print("  display(html)")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


🚀 Initializing Surveillance System...
📥 Loading buffalo_l face recognition model...
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'sdpa_kernel': '0', 'use_tf32': '1', 'fuse_conv_bias': '0', 'prefer_nhwc': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_enable': '0', 'use_ep_level_unified_stream': '0', 'device_id': '0', 'has_user_compute_stream': '0', 'gpu_external_empty_cache': '0', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'cudnn_conv1d_pad_to_nc1d': '0', 'gpu_mem_limit': '18446744073709551615', 'gpu_external_alloc': '0', 'gpu_external_free': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'do_copy_in_default_stream': '1', 'enable_cuda_graph': '0', 'user_compute_stream': '0', 'cudnn_conv_use_max_workspace': '1'}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 

📹 Video: 854x480 @ 29 FPS

✅ Processed 880 frames
🔄 Converting to H.264...
   Input: output2.mp4
   Output: output2_h264.mp4
   Quality: CRF 23, Preset: medium
   Input size: 2.2 MB
✅ Conversion successful!
   Output size: 0.7 MB
   Compression: 68.7%
📹 Displaying video: output2_h264.mp4
   File size: 0.7 MB



✅ Processing complete!

To display in Colab:
  from IPython.display import display
  html = ColabVideoHelpers.display_and_download('output.mp4')
  display(html)


#Debugging

In [ ]:
    from huggingface_hub import hf_hub_download
    config.clothing_model_path = hf_hub_download(
    repo_id="kesimeg/yolov8n-clothing-detection",
    filename="best.pt"
    )
    # Initialize
system = SurveillanceSystem()
system.create_uniform_profile("staff_uniform", "Uniform.jpg")

# 3. Create debugger
debugger = UniformColorDebugger(system)

# 4. Debug template
debugger.debug_single_image("๊Uniform.jpg")
# Debug frame ที่ 100 จากวิดีโอ
debugger.debug_video_frame("Employee_Walk.mp4", frame_number=100)
# เปรียบเทียบโดยตรง
debugger.compare_template_vs_video(
    template_path="Uniform.jpg",
    video_path="Employee_Walk.mp4",
    frame_number=100
)